# AI-RAG · Week 12 — Notebook 1 (Day 1)
## Debugging RAG Systematically

*Same symptom, different causes, different fixes — classify first.* This notebook is the staged RAG pipeline that emits a **trace**, the trace object you read to localise a failure, the four-bucket failure taxonomy, and the **six-scenario debugging drill**. It runs **fully offline and deterministically** — the pipeline is synthetic (no OpenAI, no Qdrant, no cost), so every trace is identical on every machine.

> Code is lifted **verbatim** from the package modules (`src/rag/*.py`, `src/debug/*.py`). The only changes are notebook glue (a local golden-set path, and dropping the relative-import line because the code is defined inline).

### Setup

In [1]:
# --- notebook setup (delivery glue only) -------------------------------------
# The pipeline is fully synthetic + deterministic (no OpenAI, no Qdrant, no cost),
# so this notebook runs top-to-bottom on any machine. We only write the golden set
# the pipeline reads.
from pathlib import Path
GOLDEN_JSONL = r"""{"id": "g001", "question": "What is the company's leave policy?", "ideal_answer": "Employees get 20 days of paid leave per year, accrued monthly at 1.67 days per month. Unused leave caps at 30 days; anything above that lapses on 31 December.", "notes": "happy-path. Source: HR-policy.pdf §3.1."}
{"id": "g002", "question": "How do I request a reimbursement for a work-related expense under $100?", "ideal_answer": "Submit the receipt in the expense portal within 30 days of the purchase. Items under $100 are auto-approved if they fit a standard category (travel, meals, supplies). Submission outside 30 days requires manager approval.", "notes": "happy-path. Source: Finance-policy.pdf §5.4."}
{"id": "g003", "question": "What is the standard working week?", "ideal_answer": "The standard working week is 40 hours, Monday to Friday, 09:00 to 17:30 with a one-hour lunch. Flexible start times between 08:00 and 10:00 are permitted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §1.2."}
{"id": "g004", "question": "What's the policy on bringing personal devices to work (BYOD)?", "ideal_answer": "Personal devices may be used to access company email and approved SaaS apps via the MDM enrollment. Devices must be password-protected, encrypted, and patched within 30 days of OS updates. Local storage of company documents is prohibited.", "notes": "happy-path. Source: IT-security.pdf §2.3."}
{"id": "g005", "question": "How do I request parental leave?", "ideal_answer": "Submit a parental leave request via the HR portal at least 8 weeks before the expected leave start date. Primary caregivers are entitled to 16 weeks paid; secondary caregivers to 4 weeks paid. Additional unpaid leave up to 12 months may be granted with manager approval.", "notes": "happy-path. Source: HR-policy.pdf §3.4."}
{"id": "g006", "question": "What's the dress code for client meetings?", "ideal_answer": "Business smart for in-person client meetings: collared shirt or blouse, smart trousers or skirt, closed-toe shoes. Jeans and sneakers are acceptable for internal meetings but not for client-facing ones.", "notes": "happy-path. Source: HR-policy.pdf §2.1."}
{"id": "g007", "question": "How do I report a security incident?", "ideal_answer": "Email security@company.com immediately and create a ticket in the IT portal flagged as 'Security Incident'. For incidents involving customer data, also notify the Data Protection Officer within 24 hours. Do not investigate the incident yourself — preserve evidence.", "notes": "happy-path. Source: IT-security.pdf §4.1."}
{"id": "g008", "question": "What's the process for requesting new software?", "ideal_answer": "Software requests go through the IT portal. Pre-approved software (listed in the IT catalogue) is auto-installed within 1 business day. New requests require justification, a security review (2-5 business days), and budget approval if the cost exceeds $500/year.", "notes": "happy-path. Source: IT-policy.pdf §3.2."}
{"id": "g009", "question": "Who do I contact about my pay or benefits?", "ideal_answer": "For pay queries, contact Payroll at payroll@company.com or via the HR portal under 'Pay & Benefits'. For benefits (health insurance, retirement, etc.), contact HR Benefits at benefits@company.com.", "notes": "happy-path. Source: HR-policy.pdf §6."}
{"id": "g010", "question": "What's the policy on working from home?", "ideal_answer": "Employees may work remotely up to 3 days per week with manager approval. Remote work requires a stable internet connection, a quiet workspace, and availability during core hours (10:00-15:00 local time). Full-time remote arrangements require VP-level approval.", "notes": "happy-path. Source: HR-policy.pdf §1.5."}
{"id": "g011", "question": "How do I book a meeting room?", "ideal_answer": "Meeting rooms are booked via the Outlook calendar. Rooms over 12 people require booking at least 24 hours in advance. Recurring meetings should not block premium rooms (those with video conferencing) for more than 2 hours per week.", "notes": "happy-path. Source: Facilities-policy.pdf §1.1."}
{"id": "g012", "question": "What's the company's policy on accepting gifts from clients?", "ideal_answer": "Gifts under $50 in value may be accepted with line-manager notification. Gifts above $50 must be declined or, if culturally inappropriate to decline, donated to charity with HR notified. Cash, gift cards, and personal services may never be accepted.", "notes": "happy-path. Source: Compliance-policy.pdf §2.1."}
{"id": "g013", "question": "How do I onboard a new contractor?", "ideal_answer": "Submit a contractor onboarding request via the HR portal at least 5 business days before the start date. Include scope of work, contract duration, budget code, and the contractor's NDA status. IT provisioning takes 2 business days from approval.", "notes": "happy-path. Source: HR-policy.pdf §5.1."}
{"id": "g014", "question": "What training is required for new employees?", "ideal_answer": "All new employees complete three mandatory trainings within 30 days of joining: (1) Information Security & Data Protection, (2) Code of Conduct & Anti-Harassment, (3) Health & Safety. Tracked via the L&D portal; manager is notified if not completed by day 30.", "notes": "happy-path. Source: HR-policy.pdf §4.2."}
{"id": "g015", "question": "How does the remote work policy interact with the requirement to attend in-person team meetings?", "ideal_answer": "Remote work allowance (up to 3 days per week) is subject to attending mandatory in-person team meetings as scheduled by the manager. Teams typically schedule one anchor in-person day per week. Recurring conflicts should be resolved by the team's manager, who may approve exceptions on a case-by-case basis.", "notes": "harder · multi-hop. Cross-references HR-policy §1.5 (remote) AND §2.2 (team meetings). Tests integration across two source sections."}
{"id": "g016", "question": "If I'm on parental leave and a serious security incident happens in my product area, am I expected to respond?", "ideal_answer": "No. Employees on parental leave are not expected to respond to work matters, including security incidents. The on-call rotation and the team's secondary contact (named in each team's runbook) cover incident response during any team member's absence. If the matter is critical and only the absent employee has the knowledge, the line manager escalates to the VP for coverage.", "notes": "harder · multi-hop. Cross-references HR-policy §3.4 (parental leave) AND IT-security §4.1 (incident response). Tests respect for leave + named escalation path."}
{"id": "g017", "question": "What's the right channel to report a potential conflict of interest involving a vendor my partner works for?", "ideal_answer": "Disclose the potential conflict via the Compliance portal under 'Conflict of Interest Disclosure'. Include the vendor name, your partner's role, and any business relationship you have with the vendor. The Compliance team reviews disclosures within 5 business days and may require recusal from procurement or vendor-management decisions involving that vendor.", "notes": "harder. Combines Compliance-policy §3 (disclosures) and process detail. Tests nuanced conflict-of-interest reporting."}
{"id": "g018", "question": "Can I expense a Coursera certification that's not in the L&D catalogue if it directly supports my project?", "ideal_answer": "Yes, with manager pre-approval. Submit a request via the L&D portal under 'Out-of-catalogue learning' with a one-paragraph business justification. Certifications up to $1,000 per year per employee are typically approved for project-relevant skills. Reimbursement requires successful completion (certificate issued) within 6 months.", "notes": "harder. Cross-references L&D-policy §2.4 and Finance §5. Tests handling of off-catalogue requests with conditions."}
{"id": "g019", "question": "What is the company's stance on cryptocurrency investing as part of the benefits package?", "ideal_answer": "Cryptocurrency investing is not part of the benefits package. The answer is not in the company's policy documents. The assistant should say so clearly and suggest the employee contact HR for clarification.", "notes": "edge. Tests graceful refusal — answer NOT in docs. Should not hallucinate."}
{"id": "g020", "question": "Will the company sponsor my visa for a personal trip to a country requiring a business invitation letter?", "ideal_answer": "Visa support is only provided for company-sanctioned business travel — see Travel-policy §3.2. Personal travel visas are the employee's responsibility. The company does not issue invitation letters for personal trips.", "notes": "edge. Tests boundary-setting — adjacent to a real policy but clearly outside its scope. Should redirect, not invent."}"""
Path("golden_set").mkdir(exist_ok=True)
Path("golden_set/capstone_v1.jsonl").write_text(GOLDEN_JSONL + "\n")
print("golden set written:", len(GOLDEN_JSONL.splitlines()), "questions · pipeline is offline + deterministic")


golden set written: 20 questions · pipeline is offline + deterministic


## 1 · The trace — `src/rag/trace.py`

Trace reading is *the* production skill of the week. A **Trace** is the input and output of every pipeline stage for one query. You debug by scanning it for the single stage whose output is wrong. `render()` is what you read.

In [2]:
"""src/rag/trace.py — a readable trace of one RAG query.

Trace reading is *the* production skill of W12. A trace is the input and output
of every pipeline stage for a single query: embed → retrieve → rerank → cache →
prompt → generate. You debug by scanning the trace for the one stage whose
output is wrong.

This is deliberately dependency-free (stdlib only) so the debugging drill runs
on any machine with no API key, no Qdrant, and no cost.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field, asdict


@dataclass
class StageRecord:
    """One pipeline stage's input/output, as it would appear in a real trace."""

    stage: str                       # embed | retrieve | rerank | cache | prompt | generate
    summary: str                     # one-line human-readable headline
    detail: dict = field(default_factory=dict)   # structured evidence
    ok: bool = True                  # False if the stage errored (infra failure)
    latency_ms: int = 0


@dataclass
class Trace:
    """The full trace of a single query."""

    question: str
    stages: list[StageRecord] = field(default_factory=list)
    answer: str = ""

    def add(self, stage: str, summary: str, detail: dict | None = None,
            ok: bool = True, latency_ms: int = 0) -> StageRecord:
        rec = StageRecord(stage, summary, detail or {}, ok, latency_ms)
        self.stages.append(rec)
        return rec

    def get(self, stage: str) -> StageRecord | None:
        for s in self.stages:
            if s.stage == stage:
                return s
        return None

    @property
    def total_latency_ms(self) -> int:
        return sum(s.latency_ms for s in self.stages)

    def to_json(self) -> str:
        return json.dumps(
            {"question": self.question,
             "stages": [asdict(s) for s in self.stages],
             "answer": self.answer,
             "total_latency_ms": self.total_latency_ms},
            indent=2,
        )

    def render(self) -> str:
        """Human-readable trace — this is what a learner scans for the outlier."""
        lines = [
            "─" * 68,
            f"TRACE · {self.question}",
            "─" * 68,
        ]
        for i, s in enumerate(self.stages, 1):
            flag = " " if s.ok else "✗"
            lines.append(f"[{i}] {flag} {s.stage.upper():9} {s.latency_ms:>5}ms  {s.summary}")
            for k, v in s.detail.items():
                text = v if isinstance(v, str) else json.dumps(v)
                if len(text) > 88:
                    text = text[:85] + "…"
                lines.append(f"        {k}: {text}")
        lines.append("─" * 68)
        lines.append(f"ANSWER: {self.answer}")
        lines.append(f"(total {self.total_latency_ms}ms)")
        return "\n".join(lines)


## 2 · The staged pipeline — `src/rag/pipeline.py`

Six stages — embed → cache → retrieve → rerank → prompt → generate — recording a stage in the trace at each step. `run_query(entry)` runs the healthy pipeline; `run_query(entry, scenario_id=...)` breaks **exactly one** stage so you can practise localising the failure.

In [3]:
"""src/rag/pipeline.py — a deterministic, offline, staged RAG pipeline.

Runs the six stages of a RAG query — embed → retrieve → rerank → cache → prompt
→ generate — over the capstone golden set, recording a Trace at each step. It is
fully synthetic (no OpenAI, no Qdrant, no cost) and deterministic, so the W12
debugging drill produces the *same* trace every time on every machine.

`run_query(entry)` runs the healthy pipeline. `run_query(entry, scenario_id=...)`
breaks exactly one stage so learners can practise localising the failure from
the trace. The scenario definitions live in src/debug/scenarios.py.
"""
from __future__ import annotations

import json
import re
from pathlib import Path

# Trace is defined in the cell above

GOLDEN_PATH = Path("golden_set/capstone_v1.jsonl")   # notebook: local file written in setup

# A small fixed pool of clearly off-topic chunks — the "noise" a retriever can
# wrongly pull into the top-K.
DISTRACTORS = [
    {"chunk_id": "kb-441", "text": "The office kitchen is restocked every Monday and reception opens at eight."},
    {"chunk_id": "kb-887", "text": "Visitor parking is on level 2 and requires a validation stamp from the desk."},
    {"chunk_id": "kb-203", "text": "The annual company picnic is held in July at the lakeside pavilion."},
]

# The single line that keeps generation grounded. Dropping it is a prompt failure.
GROUNDING = ("Answer ONLY using the provided context. If the answer is not in the "
             "context, say you don't know and suggest contacting the relevant team.")

_STOP = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with", "is",
         "are", "was", "were", "be", "as", "at", "by", "it", "this", "that", "from",
         "your", "you", "may", "must", "can", "do", "does", "how", "what", "who",
         "i", "my", "we", "our", "per", "up", "any", "all", "via", "not", "s"}


def _tokens(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-z0-9]+", (text or "").lower())
            if len(t) > 2 and t not in _STOP}


def _score(chunk_text: str, question: str) -> float:
    q = _tokens(question)
    if not q:
        return 0.0
    return round(len(_tokens(chunk_text) & q) / len(q), 3)


def load_golden(path: str | Path = GOLDEN_PATH) -> dict[str, dict]:
    """Return {id: entry}. Entry = {id, question, ideal_answer, notes}."""
    out: dict[str, dict] = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("//"):
            e = json.loads(line)
            out[e["id"]] = e
    return out


def _relevant_chunks(entry: dict) -> list[dict]:
    """Split the ideal answer into two 'retrieved' chunks (a two-hop answer)."""
    words = str(entry["ideal_answer"]).split()
    mid = max(1, len(words) // 2)
    return [
        {"chunk_id": f"{entry['id']}-a", "text": " ".join(words[:mid])},
        {"chunk_id": f"{entry['id']}-b", "text": " ".join(words[mid:])},
    ]


def _candidates(entry: dict) -> list[dict]:
    """All candidate chunks (relevant + distractors) scored against the question."""
    cands = _relevant_chunks(entry) + [dict(d) for d in DISTRACTORS]
    for c in cands:
        c["score"] = _score(c["text"], entry["question"])
    return cands


# ─── the pipeline ────────────────────────────────────────────────────────────
def run_query(entry: dict, scenario_id: str = "healthy",
              golden: dict | None = None) -> Trace:
    """Run the six-stage pipeline. `scenario_id` breaks exactly one stage."""
    tr = Trace(question=entry["question"])
    rel_ids = {f"{entry['id']}-a", f"{entry['id']}-b"}

    # 1 ── embed ──────────────────────────────────────────────────────────────
    emb = sorted(_tokens(entry["question"]))
    tr.add("embed", f"{len(emb)} content tokens embedded", {"tokens": emb[:6]}, latency_ms=42)

    # 2 ── cache (checked early; a hit short-circuits the whole pipeline) ──────
    golden = golden or {}
    if scenario_id == "S5":                       # stale cache: hit from another question
        stale = golden.get("g003", {"ideal_answer": "(stale)"})
        tr.add("cache", "HIT — returned cached answer",
               {"cache_hit": True, "cached_for_question": "g003",
                "note": "answer served from cache; retrieve + generate skipped"}, latency_ms=6)
        tr.answer = stale["ideal_answer"]
        return tr
    tr.add("cache", "miss — running full pipeline", {"cache_hit": False}, latency_ms=5)

    # 3 ── retrieve ────────────────────────────────────────────────────────────
    cands = _candidates(entry)
    if scenario_id == "S1":                       # recall miss: drop the 2nd hop
        cands = [c for c in cands if c["chunk_id"] != f"{entry['id']}-b"]
    elif scenario_id == "S2":                     # precision noise: distractors on top
        for c in cands:
            if c["chunk_id"].startswith("kb-"):
                c["score"] = round(0.9 - 0.05 * next(i for i, d in enumerate(DISTRACTORS)
                                                     if d["chunk_id"] == c["chunk_id"]), 3)
            else:
                c["score"] = 0.15
    retrieved = sorted(cands, key=lambda c: c["score"], reverse=True)[:3]
    tr.add("retrieve", f"top-{len(retrieved)} by similarity",
           {"chunk_ids": [c["chunk_id"] for c in retrieved],
            "scores": [c["score"] for c in retrieved]}, latency_ms=88)

    # 4 ── rerank ──────────────────────────────────────────────────────────────
    reranked = sorted(retrieved, key=lambda c: c["score"], reverse=True)
    tr.add("rerank", "cross-encoder reorder",
           {"order": [c["chunk_id"] for c in reranked],
            "top1_score": reranked[0]["score"] if reranked else 0.0}, latency_ms=61)

    # 5 ── prompt ─────────────────────────────────────────────────────────────
    grounding = "" if scenario_id == "S4" else GROUNDING   # S4 drops the instruction
    ctx = "\n".join(f"- {c['text']}" for c in reranked)
    prompt = f"{grounding}\n\nContext:\n{ctx}\n\nQuestion: {entry['question']}".strip()
    tr.add("prompt", f"prompt built ({len(prompt.split())} tokens)",
           {"has_grounding_instruction": bool(grounding),
            "prompt_preview": prompt.replace("\n", " ")[:120]}, latency_ms=3)

    # 6 ── generate ───────────────────────────────────────────────────────────
    top = reranked[0] if reranked else {"chunk_id": "", "text": ""}
    top_relevant = top["chunk_id"] in rel_ids

    if scenario_id == "S6":                        # infra: generation error/timeout
        tr.add("generate", "ERROR — model call failed",
               {"error": "OpenAI timeout after 30000ms", "retries": 0},
               ok=False, latency_ms=30000)
        tr.answer = "(no answer returned — generation error)"
        return tr

    if scenario_id == "S3":                        # generation: hallucination
        answer = (str(entry["ideal_answer"]) +
                  " Unused leave can also be cashed out at 150% of salary on request.")
        note = "answer adds a claim not present in any retrieved chunk"
    elif scenario_id == "S4":                      # prompt: no grounding → invents a step
        answer = (str(entry["ideal_answer"]) +
                  " You should also post the incident in the public #general Slack channel.")
        note = "no grounding instruction; answer includes an unsupported step"
    elif scenario_id in ("S1",):                   # recall miss → partial answer
        answer = reranked[0]["text"] if reranked else ""
        note = "only the first hop was retrieved; answer is incomplete"
    elif scenario_id in ("S2",) or not top_relevant:   # wrong chunks → wrong answer
        answer = f"According to the retrieved context: {top['text']}"
        note = "answer generated from off-topic chunks"
    else:                                          # healthy: grounded, correct
        answer = str(entry["ideal_answer"])
        note = "grounded in retrieved context"
    tr.add("generate", "answer generated",
           {"tokens": len(answer.split()), "cost_usd": 0.00021, "note": note}, latency_ms=430)
    tr.answer = answer
    return tr


**Run one healthy query and read its trace:**

In [4]:
golden = load_golden()
tr = run_query(golden['g001'], scenario_id='healthy', golden=golden)
print(tr.render())

────────────────────────────────────────────────────────────────────
TRACE · What is the company's leave policy?
────────────────────────────────────────────────────────────────────
[1]   EMBED        42ms  3 content tokens embedded
        tokens: ["company", "leave", "policy"]
[2]   CACHE         5ms  miss — running full pipeline
        cache_hit: false
[3]   RETRIEVE     88ms  top-3 by similarity
        chunk_ids: ["g001-a", "g001-b", "kb-203"]
        scores: [0.333, 0.333, 0.333]
[4]   RERANK       61ms  cross-encoder reorder
        order: ["g001-a", "g001-b", "kb-203"]
        top1_score: 0.333
[5]   PROMPT        3ms  prompt built (76 tokens)
        has_grounding_instruction: true
        prompt_preview: Answer ONLY using the provided context. If the answer is not in the context, say you …
[6]   GENERATE    430ms  answer generated
        tokens: 29
        cost_usd: 0.00021
        note: grounded in retrieved context
─────────────────────────────────────────────────────────

## 3 · The failure taxonomy — four **primary runtime** buckets

Before you debug, **classify**. "The answer is wrong" is not a diagnosis — the same symptom has different causes. Four *primary runtime* buckets localise a traced failure:

| Bucket | What it is | Diagnostic move |
|---|---|---|
| **Retrieval** | the right chunk wasn't retrieved (recall miss · precision noise) | check `retrieve`/`rerank` |
| **Generation** | right chunks, but the model answered wrong (hallucination · off-topic) | check `generate` |
| **Prompt** | the assembled prompt is broken (lost grounding line · wrong template) | check `prompt` |
| **Infra** | the plumbing failed (stale cache · timeout · API error) | check `cache`/`generate` |

These four localise a failure **from the trace**. They are not the whole universe of RAG failures: **beyond the runtime pipeline**, cross-cutting failures — ingestion/data-quality (W8), access & security, evaluation/reference defects (W11), and user-input/policy — sit upstream or across stages and are diagnosed elsewhere. Classify first; fix second — and treat fixes as **candidates you test against the golden set**, not guarantees.

## 4 · The six scenarios — `src/debug/scenarios.py`

Each scenario runs the pipeline with exactly one stage broken. The `bucket` / `root_cause` / `fix` / `trace_tell` fields are the **answer key** — revealed by the grader only after a correct classification.

In [5]:
"""src/debug/scenarios.py — the W12 debugging drill: six broken RAG queries.

Each scenario runs the pipeline with exactly one stage broken. The learner reads
the trace, classifies the failure into one of the four taxonomy buckets, and
names the fix. The bucket / root_cause / fix below are the answer key — revealed
by the drill only after a correct classification.

Four buckets (the taxonomy):
    retrieval  — the right chunk wasn't retrieved (precision or recall)
    generation — right chunks retrieved, but the LLM answered wrong / hallucinated
    prompt     — the prompt itself is wrong (e.g. missing the grounding instruction)
    infra      — the pipeline broke (timeout, API error, stale cache)
"""
from __future__ import annotations

from dataclasses import dataclass

BUCKETS = ("retrieval", "generation", "prompt", "infra")


@dataclass(frozen=True)
class Scenario:
    id: str
    title: str
    question_id: str          # which golden question to run it on
    symptom: str              # what the learner is told up front
    bucket: str               # ANSWER KEY — the correct taxonomy bucket
    root_cause: str           # ANSWER KEY — what actually went wrong
    fix: str                  # ANSWER KEY — the smallest correct fix
    trace_tell: str           # ANSWER KEY — the outlier to spot in the trace


SCENARIOS: list[Scenario] = [
    Scenario(
        id="S1",
        title="The multi-hop answer is missing half",
        question_id="g015",
        symptom="The answer covers remote-work days but says nothing about the in-person meeting requirement.",
        bucket="retrieval",
        root_cause="Recall miss: the second required chunk (the in-person-meetings hop) was never retrieved, so the model couldn't answer it.",
        fix="Raise top-K or add query expansion for multi-hop questions so both required chunks reach the context.",
        trace_tell="In retrieve, only the '-a' chunk is present; the '-b' (second hop) chunk is absent and distractors fill the top-K.",
    ),
    Scenario(
        id="S2",
        title="The answer is about the office kitchen",
        question_id="g004",
        symptom="The answer is fluent but talks about kitchens/parking instead of the BYOD policy that was asked about.",
        bucket="retrieval",
        root_cause="Precision failure: distractor chunks were scored highest and pushed the correct chunk out of the top-K, so generation had only noise to work from.",
        fix="Retune hybrid weighting / the reranker (or reduce chunk size) so relevant chunks outrank distractors.",
        trace_tell="In retrieve/rerank, the top chunk_ids are 'kb-*' distractors with high scores; the relevant chunk is ranked last or missing.",
    ),
    Scenario(
        id="S3",
        title="A confident fact that isn't in the docs",
        question_id="g001",
        symptom="The answer is mostly right but adds a specific claim (a 150%-of-salary cash-out) that no policy contains.",
        bucket="generation",
        root_cause="Hallucination: retrieval was correct, but the model asserted a fact absent from the retrieved context.",
        fix="Lower temperature and/or tighten the prompt's grounding constraint; the retrieval layer is fine.",
        trace_tell="retrieve is healthy (correct '-a'/'-b' chunks, good scores) but the answer contains a claim not present in any chunk.",
    ),
    Scenario(
        id="S4",
        title="An extra step nobody asked for",
        question_id="g007",
        symptom="The security-incident answer is correct but tacks on 'post it in the public #general Slack channel', which is not policy.",
        bucket="prompt",
        root_cause="The prompt template lost its 'answer only from context' instruction, so the model felt free to add unsupported steps.",
        fix="Restore the grounding instruction in the prompt template.",
        trace_tell="In prompt, has_grounding_instruction is false; the answer then includes content not in the context.",
    ),
    Scenario(
        id="S5",
        title="A fluent answer to a different question",
        question_id="g010",
        symptom="Asked about working from home, the answer describes standard working hours instead — fluent, but off-topic.",
        bucket="infra",
        root_cause="Stale cache: a cache hit returned a previously-cached answer for a different question (key collision / not invalidated), so retrieve and generate never ran.",
        fix="Fix the cache key / invalidate stale entries (tie invalidation to the W10 tombstone + kb_version flow).",
        trace_tell="cache shows cache_hit=true with cached_for_question=g003, and the retrieve/generate stages are skipped entirely.",
    ),
    Scenario(
        id="S6",
        title="No answer at all",
        question_id="g008",
        symptom="The request returns an error instead of an answer, and it was slow before failing.",
        bucket="infra",
        root_cause="Infrastructure failure: the generation model call timed out (30s) and there was no retry, so nothing came back.",
        fix="Add a retry with backoff and a timeout budget; check model/API health and latency.",
        trace_tell="The generate stage is marked failed (✗) with error 'timeout after 30000ms' and a 30000ms latency.",
    ),
]

_BY_ID = {s.id: s for s in SCENARIOS}


def get(scenario_id: str) -> Scenario:
    if scenario_id not in _BY_ID:
        raise KeyError(f"unknown scenario {scenario_id!r}; try one of {list(_BY_ID)}")
    return _BY_ID[scenario_id]


def grade(scenario_id: str, guess_bucket: str) -> tuple[bool, str]:
    """Check a learner's bucket classification. Returns (correct, message)."""
    s = get(scenario_id)
    guess = (guess_bucket or "").strip().lower()
    if guess not in BUCKETS:
        return False, f"'{guess_bucket}' is not a bucket. Pick one of: {', '.join(BUCKETS)}."
    if guess == s.bucket:
        return True, (f"✓ Correct — {s.bucket}.\n"
                      f"  Root cause: {s.root_cause}\n"
                      f"  Fix:        {s.fix}")
    return False, (f"✗ Not {guess}. Re-read the trace — hint: look at the "
                   f"'{_hint_stage(s.bucket)}' stage.")


def _hint_stage(bucket: str) -> str:
    return {"retrieval": "retrieve/rerank", "generation": "generate",
            "prompt": "prompt", "infra": "cache/generate"}[bucket]


## 5 · The drill — read the trace, classify, check

The loop, per scenario: see the symptom → read the full trace → classify the bucket → check. First, the six symptoms:

In [6]:
for s in SCENARIOS:
    print(f"{s.id}  {s.title}")
    print(f"     symptom: {s.symptom}")

S1  The multi-hop answer is missing half
     symptom: The answer covers remote-work days but says nothing about the in-person meeting requirement.
S2  The answer is about the office kitchen
     symptom: The answer is fluent but talks about kitchens/parking instead of the BYOD policy that was asked about.
S3  A confident fact that isn't in the docs
     symptom: The answer is mostly right but adds a specific claim (a 150%-of-salary cash-out) that no policy contains.
S4  An extra step nobody asked for
     symptom: The security-incident answer is correct but tacks on 'post it in the public #general Slack channel', which is not policy.
S5  A fluent answer to a different question
     symptom: Asked about working from home, the answer describes standard working hours instead — fluent, but off-topic.
S6  No answer at all
     symptom: The request returns an error instead of an answer, and it was slow before failing.


**Run S1's trace** (a multi-hop *recall miss*) and scan for the outlier:

In [7]:
s = get('S1')
tr = run_query(golden[s.question_id], scenario_id=s.id, golden=golden)
print(f"### {s.id} — {s.title}  (question {s.question_id})")
print('SYMPTOM:', s.symptom, '\n')
print(tr.render())

### S1 — The multi-hop answer is missing half  (question g015)
SYMPTOM: The answer covers remote-work days but says nothing about the in-person meeting requirement. 

────────────────────────────────────────────────────────────────────
TRACE · How does the remote work policy interact with the requirement to attend in-person team meetings?
────────────────────────────────────────────────────────────────────
[1]   EMBED        42ms  9 content tokens embedded
        tokens: ["attend", "interact", "meetings", "person", "policy", "remote"]
[2]   CACHE         5ms  miss — running full pipeline
        cache_hit: false
[3]   RETRIEVE     88ms  top-3 by similarity
        chunk_ids: ["g015-a", "kb-441", "kb-887"]
        scores: [0.556, 0.0, 0.0]
[4]   RERANK       61ms  cross-encoder reorder
        order: ["g015-a", "kb-441", "kb-887"]
        top1_score: 0.556
[5]   PROMPT        3ms  prompt built (94 tokens)
        has_grounding_instruction: true
        prompt_preview: Answer ONLY using

**Classify it** — the grader reveals the root cause + fix on a correct bucket:

In [8]:
ok, msg = grade('S1', 'retrieval')
print(msg)

# try a wrong one to see the hint:
print()
print(grade('S1', 'generation')[1])

✓ Correct — retrieval.
  Root cause: Recall miss: the second required chunk (the in-person-meetings hop) was never retrieved, so the model couldn't answer it.
  Fix:        Raise top-K or add query expansion for multi-hop questions so both required chunks reach the context.

✗ Not generation. Re-read the trace — hint: look at the 'retrieve/rerank' stage.


**Your turn:** run any scenario and grade it. E.g. S3 (a hallucinated fact) — is it retrieval, generation, prompt, or infra?

In [9]:
s = get('S3')
print(run_query(golden[s.question_id], scenario_id=s.id, golden=golden).render())
print()
print(grade('S3', 'generation')[1])   # <- change the bucket and re-run to test yourself

────────────────────────────────────────────────────────────────────
TRACE · What is the company's leave policy?
────────────────────────────────────────────────────────────────────
[1]   EMBED        42ms  3 content tokens embedded
        tokens: ["company", "leave", "policy"]
[2]   CACHE         5ms  miss — running full pipeline
        cache_hit: false
[3]   RETRIEVE     88ms  top-3 by similarity
        chunk_ids: ["g001-a", "g001-b", "kb-203"]
        scores: [0.333, 0.333, 0.333]
[4]   RERANK       61ms  cross-encoder reorder
        order: ["g001-a", "g001-b", "kb-203"]
        top1_score: 0.333
[5]   PROMPT        3ms  prompt built (76 tokens)
        has_grounding_instruction: true
        prompt_preview: Answer ONLY using the provided context. If the answer is not in the context, say you …
[6]   GENERATE    430ms  answer generated
        tokens: 42
        cost_usd: 0.00021
        note: answer adds a claim not present in any retrieved chunk
────────────────────────────────

**All six at a glance** — each breaks one stage; each maps to one bucket:

In [10]:
for s in SCENARIOS:
    tr = run_query(golden[s.question_id], scenario_id=s.id, golden=golden)
    print(f"{s.id}  [{s.bucket:10}]  {s.title}   ({len(tr.stages)} stages)")

S1  [retrieval ]  The multi-hop answer is missing half   (6 stages)
S2  [retrieval ]  The answer is about the office kitchen   (6 stages)
S3  [generation]  A confident fact that isn't in the docs   (6 stages)
S4  [prompt    ]  An extra step nobody asked for   (6 stages)
S5  [infra     ]  A fluent answer to a different question   (2 stages)
S6  [infra     ]  No answer at all   (6 stages)


## 6 · The debugging workflow — five steps

The drill is a rehearsal for this loop, which works on **any** non-deterministic system:

1. **Reproduce** — run the same input. Repeats? If not, it's flaky → infra or temperature.
2. **Narrow** — get the trace: the input and output of every stage.
3. **Isolate** — find the *first* stage whose output is wrong → that's the bucket.
4. **Fix** — apply the **smallest** change for that bucket, **one thing only**. Retrieval changes (raise top-K, smaller chunks) each **trade off** (more top-K = more noise/cost; smaller chunks can fragment evidence), so treat them as candidates you **test against the golden set**. And no layer is unconditionally "fine" — if generation looks wrong but the evidence was insufficient or ambiguous, the real fix is retrieval.
5. **Verify** — re-run the **whole** eval suite. Green, no regressions? Done.

## 7 · The trace pays double — cost & latency

The trace already carries every query's cost and per-stage latency (no extra tooling), which is perfect for spotting a single **outlier**. But percentiles are a property of a *distribution*: **one trace = one latency**; the mean cost/query and p50/p95 come from **aggregating many traces**.

In [11]:
h = run_query(golden['g001'], scenario_id='healthy', golden=golden)
gen = h.get('generate')
print('this ONE query  → cost:', gen.detail['cost_usd'], 'USD | total latency:', h.total_latency_ms, 'ms')
print('per-stage latency (spot the outlier):')
for st in h.stages:
    print(f'   {st.stage:9} {st.latency_ms:>6} ms')
print('\n→ mean cost/query and p50/p95 need MANY such traces aggregated — not this one.')

this ONE query  → cost: 0.00021 USD | total latency: 629 ms
per-stage latency (spot the outlier):
   embed         42 ms
   cache          5 ms
   retrieve      88 ms
   rerank        61 ms
   prompt         3 ms
   generate     430 ms

→ mean cost/query and p50/p95 need MANY such traces aggregated — not this one.


## Answer key (instructors)

The full key for all six scenarios:

In [12]:
for s in SCENARIOS:
    print(f"{s.id} — {s.title}")
    print(f"   bucket:     {s.bucket}")
    print(f"   trace tell: {s.trace_tell}")
    print(f"   fix:        {s.fix}\n")

S1 — The multi-hop answer is missing half
   bucket:     retrieval
   trace tell: In retrieve, only the '-a' chunk is present; the '-b' (second hop) chunk is absent and distractors fill the top-K.
   fix:        Raise top-K or add query expansion for multi-hop questions so both required chunks reach the context.

S2 — The answer is about the office kitchen
   bucket:     retrieval
   trace tell: In retrieve/rerank, the top chunk_ids are 'kb-*' distractors with high scores; the relevant chunk is ranked last or missing.
   fix:        Retune hybrid weighting / the reranker (or reduce chunk size) so relevant chunks outrank distractors.

S3 — A confident fact that isn't in the docs
   bucket:     generation
   trace tell: retrieve is healthy (correct '-a'/'-b' chunks, good scores) but the answer contains a claim not present in any chunk.
   fix:        Lower temperature and/or tighten the prompt's grounding constraint; the retrieval layer is fine.

S4 — An extra step nobody asked for
   bu

## Recap

Classify from the trace into one of the four **runtime** buckets, isolate the first wrong stage, apply the smallest change and **verify against the golden set**. Next, **Notebook 2** packages six weeks of Phase 2 into the M2 milestone and prepares the DR #2 defense.